In [1]:
# import pandas as pd

# df = pd.read_csv('backend_app/training_data/.csv')
# df.head()

In [2]:
# type(df['timestamp'][0])

In [3]:
# df['timestamp'] = pd.to_datetime(df['timestamp'])
# df['time_diff'] = df.groupby('user_id')['timestamp'].diff().dt.total_seconds()
# mean_time_diff = df['time_diff'].mean()  # Or use .median() for the median
# df['time_diff'] = df.groupby('user_id')['timestamp'].diff().dt.total_seconds().fillna(mean_time_diff)

# df['rolling_avg_amount'] = df.groupby('user_id')['amount'].rolling(window=7, min_periods=1).mean().reset_index(0, drop=True)
# df['transactions_per_hour'] = df.groupby('user_id')['timestamp'].transform(lambda x: x.diff().dt.total_seconds().count() / 3600)


In [4]:
# df.head()

In [5]:
# df.isnull().sum()

In [6]:
# df['user_id'].value_counts()

In [15]:
from sklearn.model_selection import train_test_split
import pandas as pd
def split_data(df_clean):

    Y = df_clean['is_fraudulent']
    X = df_clean.drop(columns=['is_fraudulent'], axis=1)   
    X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.1, random_state=42, stratify=Y)

    # Print dataset sizes
    print(f"Training Set: {X_train.shape[0]} rows")
    print(f"Testing Set: {X_test.shape[0]} rows")  

    return X_train, X_test, y_train, y_test 

df_clean = pd.read_csv('../backend_app/training_data/clean_dataset_2.csv')
df_clean.drop(columns=['transaction_velocity', 'hour', 'lon', "transaction_ratio"], inplace=True)
X_train, X_test, y_train, y_test = split_data(df_clean)
print(f"X_train : {X_train.shape}")
print(f"X_train : {X_test.shape}")


Training Set: 450000 rows
Testing Set: 50000 rows
X_train : (450000, 4)
X_train : (50000, 4)


In [8]:
df_clean.columns

Index(['transaction_type', 'amount', 'merchant_category', 'is_fraudulent',
       'transaction_velocity', 'rolling_avg_amount', 'hour', 'lon',
       'transaction_ratio'],
      dtype='object')

In [16]:
X_train.head()

,transaction_type,amount,merchant_category,rolling_avg_amount
495196,2,0.949422,5,0.271471
389655,2,0.405989,5,0.051091
171166,0,0.410042,0,0.010669
76327,2,0.655199,4,0.013974
492418,2,0.830361,5,0.181044


In [27]:
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

# Assuming X_train and X_test are available
# Load data if needed (Uncomment the following lines if required)
# df = pd.read_csv("your_data.csv")
# X = df.drop(columns=['is_fraudulent'])  # Features
# y = df['is_fraudulent']  # Target
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

# Standardizing the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Define hyperparameters for GridSearchCV
param_grid = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
    'gamma': ['scale', 'auto'],
    'degree': [2, 3, 4]  # Only relevant for 'poly' kernel
}

# Initialize SVM model
svm = SVC()

# Perform Grid Search with cross-validation
grid_search = GridSearchCV(svm, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=2)
grid_search.fit(X_train_scaled, y_train)

# Best model
best_model = grid_search.best_estimator_
print(f"Best parameters: {grid_search.best_params_}")

# Evaluate model on test set
y_pred = best_model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.4f}")
print("Classification Report:")
print(classification_report(y_test, y_pred))


Fitting 5 folds for each of 96 candidates, totalling 480 fits
[CV] END ........C=0.1, degree=2, gamma=scale, kernel=linear; total time=   9.6s
[CV] END ........C=0.1, degree=2, gamma=scale, kernel=linear; total time=  10.1s
[CV] END ........C=0.1, degree=2, gamma=scale, kernel=linear; total time=  10.1s
[CV] END ........C=0.1, degree=2, gamma=scale, kernel=linear; total time=  10.2s
[CV] END ........C=0.1, degree=2, gamma=scale, kernel=linear; total time=  10.3s
[CV] END ...........C=0.1, degree=2, gamma=scale, kernel=rbf; total time=  26.8s
[CV] END ...........C=0.1, degree=2, gamma=scale, kernel=rbf; total time=  27.4s
[CV] END .......C=0.1, degree=2, gamma=scale, kernel=sigmoid; total time=  21.1s
[CV] END ...........C=0.1, degree=2, gamma=scale, kernel=rbf; total time=  24.4s
[CV] END ...........C=0.1, degree=2, gamma=scale, kernel=rbf; total time=  24.3s
[CV] END ...........C=0.1, degree=2, gamma=scale, kernel=rbf; total time=  24.4s
[CV] END .......C=0.1, degree=2, gamma=scale, k

Best parameters: {'C': 0.1, 'degree': 2, 'gamma': 'scale', 'kernel': 'linear'}

In [17]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

In [31]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

# Assuming X_train, X_test, y_train, y_test are already defined
# Standardize the data (optional for Decision Trees)
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)

### 🚀 Decision Tree with GridSearchCV ###
dt_params = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 5]
}

dt_model = DecisionTreeClassifier(random_state=42)
dt_grid = GridSearchCV(dt_model, dt_params, cv=3, scoring='accuracy', n_jobs=-1, verbose=2)
dt_grid.fit(X_train, y_train)

print("Best Decision Tree Parameters:", dt_grid.best_params_)
best_dt = dt_grid.best_estimator_

# 🎯 Evaluate Decision Tree
y_pred_dt = best_dt.predict(X_test)
print(f"Decision Tree Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}")
print("Decision Tree Classification Report:")
print(classification_report(y_test, y_pred_dt))


Fitting 3 folds for each of 72 candidates, totalling 216 fits
[CV] END criterion=gini, max_depth=None, min_samples_leaf=1, min_samples_split=2; total time=   0.4s
[CV] END criterion=gini, max_depth=None, min_samples_leaf=1, min_samples_split=2; total time=   0.5s
[CV] END criterion=gini, max_depth=None, min_samples_leaf=1, min_samples_split=2; total time=   0.7s
[CV] END criterion=gini, max_depth=None, min_samples_leaf=1, min_samples_split=5; total time=   0.8s
[CV] END criterion=gini, max_depth=None, min_samples_leaf=1, min_samples_split=5; total time=   0.9s
[CV] END criterion=gini, max_depth=None, min_samples_leaf=1, min_samples_split=5; total time=   0.9s
[CV] END criterion=gini, max_depth=None, min_samples_leaf=1, min_samples_split=10; total time=   0.9s
[CV] END criterion=gini, max_depth=None, min_samples_leaf=1, min_samples_split=10; total time=   0.9s
[CV] END criterion=gini, max_depth=None, min_samples_leaf=2, min_samples_split=2; total time=   0.8s
[CV] END criterion=gini, ma

Best Decision Tree Parameters: {'criterion': 'gini', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2}


Best Decision Tree Parameters: {'criterion': 'gini', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2}


In [ ]:
{'criterion': 'gini', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2}


In [14]:
X_train.head()

,transaction_type,amount,merchant_category,transaction_velocity,rolling_avg_amount,hour,lon,transaction_ratio
495196,2,0.949422,5,0.578512,0.271471,5,0.876695,0.440214
389655,2,0.405989,5,0.061030,0.051091,23,0.887152,0.010633
171166,0,0.410042,0,0.176732,0.010669,5,0.204897,0.008261
76327,2,0.655199,4,0.503497,0.013974,17,0.139648,0.059430
492418,2,0.830361,5,0.654164,0.181044,5,0.768446,0.440214


## Train DT Model

In [18]:
### 🚀 Train Decision Tree Model with Given Parameters ###
dt_model = DecisionTreeClassifier(
    criterion='gini',
    max_depth=None,
    min_samples_leaf=1,
    min_samples_split=2,
    random_state=42
)

dt_model.fit(X_train, y_train)

# 🎯 Evaluate Decision Tree
y_pred_dt = dt_model.predict(X_test)
print(f"Decision Tree Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}")
print("Decision Tree Classification Report:")
print(classification_report(y_test, y_pred_dt))

Decision Tree Accuracy: 1.0000
Decision Tree Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     25000
           1       1.00      1.00      1.00     25000

    accuracy                           1.00     50000
   macro avg       1.00      1.00      1.00     50000
weighted avg       1.00      1.00      1.00     50000



In [19]:
import joblib

# Save the trained model
joblib.dump(dt_model, '../backend_app/trained_model_weights/decision_tree_model.pkl')

print("Model saved as random_forest_model.pkl")


Model saved as random_forest_model.pkl


In [31]:
import numpy as np
import pandas as pd
import shap
import joblib
# import lime.lime_tabular

from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

def predict_fraud_with_dt(values, dt_model):
    merchant_category = {"groceries": 0, "bank": 1, "electronics": 2, "atm": 3, "restaurant": 4, "luxury goods": 5}
    values['merchant_category'] = merchant_category[values['merchant_category'].lower()]
    transaction_type = {"refund" : 0,  "transfer": 1, "purchase": 2, "withdrawal": 3}

    values['transaction_type'] = transaction_type[values['transaction_type'].lower()]
    values['amount'] = np.log1p(values['amount'])

    new_transaction = pd.DataFrame([values])

    columns_to_normalize = ['amount']
    new_transaction[columns_to_normalize] = scaler.fit_transform(new_transaction[columns_to_normalize])

    print("Prediction time....")
    # fraud_probability = model.predict(new_transaction)[0][0]
    prediction = dt_model.predict(new_transaction)[0]
    fraud_probability = dt_model.predict_proba(new_transaction)[:, 1][0] 
    print("fraud_probability : ", fraud_probability)
    is_fraud = 1 if fraud_probability > 0.5 else 0

    print(f"Fraud Probability: {fraud_probability:.4f}")
    print(f"Fraud prediction: {prediction:.4f}")
    print(f"Is Fraudulent? {'Yes' if is_fraud else 'No'}")

    # **SHAP Explanation**
    explainer = shap.Explainer(dt_model, new_transaction)
    shap_values = explainer(new_transaction)
    # Extracting specific values
    shap_values_array = shap_values.values
    base_values = shap_values.base_values
    data_values = shap_values.data

    features_importance = {"shap_values" : shap_values_array, "base_values" : base_values, "data_values" : data_values}
    print("SHAP Explanation:")
    print(features_importance)
    # shap.summary_plot(shap_values, new_transaction)

    return {"fraud_probability" : fraud_probability, "verdict" :  is_fraud, "features_importance" : features_importance}

In [27]:
X_train.head()

,transaction_type,amount,merchant_category,rolling_avg_amount
495196,2,0.949422,5,0.271471
389655,2,0.405989,5,0.051091
171166,0,0.410042,0,0.010669
76327,2,0.655199,4,0.013974
492418,2,0.830361,5,0.181044


In [34]:
# # Perform inference (prediction)
# prediction = dt_model.predict(new_data_scaled)[0]
# prediction_probability = dt_model.predict_proba(new_data_scaled)[:, 1][0] 

# predict_fraud()

new_data = {
        'transaction_type': "Refund",
        'amount': 145.75,  
        'merchant_category': 'Groceries',
        "rolling_avg_amount" : 0.051
    }

dt_model = joblib.load('../backend_app/trained_model_weights/decision_tree_model.pkl')
# rf_model = joblib.load('random_forest_model.pkl')

response = predict_fraud_with_dt(new_data, dt_model)
print(response)

Prediction time....
fraud_probability :  0.0
Fraud Probability: 0.0000
Fraud prediction: 0.0000
Is Fraudulent? No
SHAP Explanation:
{'shap_values': array([[[0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.]]]), 'base_values': array([[1., 0.]]), 'data_values': array([[0.   , 0.   , 0.   , 0.051]])}
{'fraud_probability': 0.0, 'verdict': 0, 'features_importance': {'shap_values': array([[[0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.]]]), 'base_values': array([[1., 0.]]), 'data_values': array([[0.   , 0.   , 0.   , 0.051]])}}


In [38]:
merchant_category = {"groceries": 0, "bank": 1, "electronics": 2, "atm": 3, "restaurant": 4, "luxury goods": 5}
transaction_type = {"refund" : 0,  "transfer": 1, "purchase": 2, "withdrawal": 3}
values = {
        'transaction_type': "transfer",
        'amount': 145.75,  
        'merchant_category': 'Groceries',
        "rolling_avg_amount" : 0.051
    }
new_transaction = pd.DataFrame([values])
# new_transaction['merchant_category'] = new_transaction['merchant_category'].map(merchant_category)
new_transaction['merchant_category'] = merchant_category[values['merchant_category'].lower()]
new_transaction['transaction_type'] = transaction_type[values['transaction_type'].lower()]

print(new_transaction)

   transaction_type  amount  merchant_category  rolling_avg_amount
0                 1  145.75                  0               0.051


In [ ]:
# Perform inference (prediction)
predictions = dt_model.predict()
prediction_probabilities = dt_model.predict_proba(new_data_scaled)[:, 1]  # Probabilities for class 1

# Convert results to a DataFrame
results = pd.DataFrame({
    "Prediction": predictions,
    "Fraud Probability": prediction_probabilities
})

## RandomForest

In [5]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

# Assuming X_train, X_test, y_train, y_test are already defined
# Standardize the data (optional for Random Forest)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### 🚀 Random Forest with GridSearchCV ###
rf_params = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

rf_model = RandomForestClassifier(random_state=42)
rf_grid = GridSearchCV(rf_model, rf_params, cv=3, scoring='accuracy', n_jobs=-1, verbose=2)
rf_grid.fit(X_train_scaled, y_train)

print("Best Random Forest Parameters:", rf_grid.best_params_)
best_rf = rf_grid.best_estimator_

# 🎯 Evaluate Random Forest
y_pred_rf = best_rf.predict(X_test_scaled)
print(f"Random Forest Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print("Random Forest Classification Report:")
print(classification_report(y_test, y_pred_rf))

Fitting 3 folds for each of 108 candidates, totalling 324 fits
[CV] END bootstrap=True, max_depth=None, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=  31.1s
[CV] END bootstrap=True, max_depth=None, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=  31.3s
[CV] END bootstrap=True, max_depth=None, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=  31.3s
[CV] END bootstrap=True, max_depth=None, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=  31.2s
[CV] END bootstrap=True, max_depth=None, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=  31.4s
[CV] END bootstrap=True, max_depth=None, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=  31.4s
[CV] END bootstrap=True, max_depth=None, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=  58.9s
[CV] END bootstrap=True, max_depth=None, min_samples_leaf=1, min_samples_split=5, n_estimators=200; tota

Best Random Forest Parameters: {'bootstrap': True, 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}

## Train Best RandomForset Model

In [6]:

### 🚀 Train Random Forest Model with Given Parameters ###
rf_model = RandomForestClassifier(
    bootstrap=True,
    max_depth=None,
    min_samples_leaf=1,
    min_samples_split=2,
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

# 🎯 Evaluate Random Forest
y_pred_rf = rf_model.predict(X_test)
print(f"Random Forest Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print("Random Forest Classification Report:")
print(classification_report(y_test, y_pred_rf))

Random Forest Accuracy: 1.0000
Random Forest Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     24956
           1       1.00      1.00      1.00     24945

    accuracy                           1.00     49901
   macro avg       1.00      1.00      1.00     49901
weighted avg       1.00      1.00      1.00     49901



In [8]:
import joblib

# Save the trained model
joblib.dump(rf_model, 'backend_app/trained_model_weights/random_forest_model.pkl')

print("Model saved as random_forest_model.pkl")


Model saved as random_forest_model.pkl


In [17]:
X_train.columns

Index(['transaction_type', 'amount', 'merchant_category', 'rolling_avg_amount',
       'lon', 'hour_of_day', 'transaction_ratio'],
      dtype='object')

In [11]:
dt_model = joblib.load('backend_app/trained_model_weights/random_forest_model.pkl')

In [44]:
new_data = {
        "transaction_type": 3,
        "amount": 145.75,  
        "merchant_category": 'groceries',
        "rolling_avg_amount" : 146,
        "lon": -102.4194,  
        "hour_of_day": 12,  
        "transaction_ratio": 0.15247458 ,
    }

merchant_category = {"groceries": 0, "bank": 1, "electronics": 2, "atm": 3, "restaurant": 4, "luxury goods": 5}
new_data['merchant_category'] = merchant_category[new_data['merchant_category']]
new_data['amount'] = np.log1p(new_data['amount'])

new_transaction = pd.DataFrame([new_data])

columns_to_normalize = ['amount', 'lon', 'rolling_avg_amount', 'transaction_ratio']
new_transaction[columns_to_normalize] = scaler.fit_transform(new_transaction[columns_to_normalize])
# fraud_probability = dt_model.predict(new_transaction)[0]
fraud_probability = rf_model.predict_proba(new_transaction)[:, 1][0]
print(fraud_probability)

is_fraud = 1 if fraud_probability > 0.5 else 0
print(is_fraud)

0.0
0


In [60]:
def insert_at_index(d, index, key, value):
    items = list(d.items())  # Convert dict to list of tuples
    items.insert(index, (key, value))  # Insert new key-value pair at index
    return dict(items)  # Convert back to dictionary

# Original dictionary
data = {
    "transaction_type": 3,
    "amount": 145.75,  
    "merchant_category": "groceries",
    "hour": 12,  
    "lon": -102.4194,  
    "transaction_ratio": 0.15247458 
}

# Insert "rolling_avg_amount" at index 3
updated_data = insert_at_index(data, 3, "rolling_avg_amount", 146)

print(updated_data)

{'transaction_type': 3, 'amount': 145.75, 'merchant_category': 'groceries', 'rolling_avg_amount': 146, 'hour': 12, 'lon': -102.4194, 'transaction_ratio': 0.15247458}


In [37]:
import pandas as pd 
import shap
import numpy as np
from traceback import format_exc


def generate_shap_description(shap_values, feature_names, base_value, prediction):
    description = f"The model predicts a {prediction * 100:.2f}% probability of fraud. Key factors contributing to this prediction are:"
    contributions = []
    print(f"shap_values : {shap_values}")
    shap_values = shap_values.flatten()
    for feature, value in zip(feature_names, shap_values):
        print(feature)
        print("value",value)
        if value == 0.0:
            contributions.append(f"The {feature} dose not contributed the likelihood of fraud by {abs(value):.2f}.")
        
        elif value > 0.0:
            contributions.append(f"The {feature} increased the likelihood of fraud by {abs(value):.2f}.")
        
        else:
            contributions.append(f"The {feature} decreased the likelihood of fraud by {abs(value):.2f}.")
    
    return description, contributions

def predict_fraud_with_rf(new_transaction, rf_model):

    try:
        print("predict_fraud_with dt ")

        # prediction = rf_model.predict(new_transaction)[0]
        fraud_probability = rf_model.predict_proba(new_transaction)[:, 1][0] 
        is_fraud = 1 if fraud_probability > 0.5 else 0

        print(f"Fraud Probability: {fraud_probability:.4f}")
        # print(f"Fraud prediction: {prediction:.4f}")
        print(f"Is Fraudulent? {'Yes' if is_fraud else 'No'}")

        # **SHAP Explanation**
        explainer = shap.Explainer(rf_model, new_transaction)
        shap_values = explainer(new_transaction)
        # Extracting specific values
        shap_values_array = shap_values.values
        base_values = shap_values.base_values
        data_values = shap_values.data 

        feature_names = new_transaction.columns.tolist()

        description, contributions = generate_shap_description(shap_values_array, feature_names, base_values, fraud_probability)

        features_importance = {"shap_values" : shap_values_array, "base_values" : base_values, "data_values" : data_values, "description" : description, "features_contributions" : contributions}

        print("SHAP Explanation:")
        print(features_importance)
        shap.summary_plot(shap_values, new_transaction)

        return {"fraud_probability" : fraud_probability, "verdict" :  is_fraud, "features_importance" : features_importance}
    
    except Exception as e:
        print(f"Error in  predict_fraud_with_dt : {format_exc()}")
        return 0.2, None 

In [ ]:

data = {
        "transaction_type": 3,
        "amount": 145.75,  
        "merchant_category": 'groceries',
        "rolling_avg_amount" : 146,
        "lon": -102.4194,  
        "hour_of_day": 12,  
        "transaction_ratio": 0.15247458 ,
    }

# Insert "rolling_avg_amount" at index 3
updated_data = insert_at_index(data, 3, "rolling_avg_amount", 146)

# print(updated_data)


merchant_category = {"groceries": 0, "bank": 1, "electronics": 2, "atm": 3, "restaurant": 4, "luxury goods": 5}
updated_data['merchant_category'] = merchant_category[updated_data['merchant_category']]
updated_data['amount'] = np.log1p(updated_data['amount'])
print(updated_data)
new_transaction = pd.DataFrame([updated_data])

columns_to_normalize = ['amount', 'lon', 'rolling_avg_amount', 'transaction_ratio']
new_transaction[columns_to_normalize] = scaler.fit_transform(new_transaction[columns_to_normalize])
rf_model = joblib.load('backend_app/trained_model_weights/random_forest_model.pkl')

predict_fraud_with_rf(new_transaction, rf_model)

{'transaction_type': 3, 'amount': 4.988730458708206, 'merchant_category': 0, 'rolling_avg_amount': 146, 'lon': -102.4194, 'hour_of_day': 12, 'transaction_ratio': 0.15247458}
predict_fraud_with_rf
Fraud Probability: 0.0000
Is Fraudulent? No


{'fraud_probability': 0.0, 'verdict': 0}

In [45]:
import pandas as pd 
import shap
import numpy as np
from traceback import format_exc

def generate_shap_description(shap_values, feature_names, base_value, prediction):
    
    description = f"The model predicts a {prediction * 100:.2f}% probability of fraud. Key factors contributing to this prediction are:"
    contributions = []
    print(f"shap_values : {shap_values}")
    try:
        # Ensure shap_values is 1D
        shap_values = shap_values.flatten()
        
        for feature, value in zip(feature_names, shap_values):
            print(feature)
            print("value", value)
            if value == 0.0:
                contributions.append(f"The {feature} does not contribute to the likelihood of fraud.")
            elif value > 0.0:
                contributions.append(f"The {feature} increased the likelihood of fraud by {abs(value):.2f}.")
            else:
                contributions.append(f"The {feature} decreased the likelihood of fraud by {abs(value):.2f}.")
        
        return description, contributions
    except Exception as e:
        print(f"Error in generate report : {format_exc()}")
        return description, contributions
    
def predict_fraud_with_rf(new_transaction, rf_model):
    try:
        print("predict_fraud_with_rf")

        # Predict fraud probability
        fraud_probability = rf_model.predict_proba(new_transaction)[:, 1][0] 
        is_fraud = 1 if fraud_probability > 0.5 else 0

        print(f"Fraud Probability: {fraud_probability:.4f}")
        print(f"Is Fraudulent? {'Yes' if is_fraud else 'No'}")

        # SHAP Explanation
        explainer = shap.Explainer(rf_model, new_transaction)
        shap_values = explainer(new_transaction)
        
        # Extracting specific values
        # shap_values_array = shap_values.values[0]  # Take the first instance
        # base_values = shap_values.base_values[0]  # Take the first instance
        # data_values = shap_values.data[0]  # Take the first instance

        # feature_names = new_transaction.columns.tolist()

        # description, contributions = generate_shap_description(shap_values_array, feature_names, base_values, fraud_probability)

        # features_importance = {
        #     "shap_values": shap_values_array,
        #     "base_values": base_values,
        #     "data_values": data_values,
        #     "description": description,
        #     "features_contributions": contributions
        # }

        # print("SHAP Explanation:")
        # print(features_importance)
        # 
        # Use force plot for single instance explanation
        # shap.force_plot(base_values, shap_values_array, new_transaction.iloc[0], matplotlib=True)

        return {
            "fraud_probability": fraud_probability,
            "verdict": is_fraud
            # "features_importance": features_importance
        }
    
    except Exception as e:
        print(f"Error in predict_fraud_with_rf: {format_exc()}")
        return {
            "fraud_probability": 0.2,
            "verdict": None,
            "features_importance": None
        }

In [56]:

predict_fraud_with_rf(updated_data, rf_model)

predict_fraud_with_rf
Error in predict_fraud_with_rf: Traceback (most recent call last):
  File "/var/folders/9b/p02j0s991xl5svg4nyn2wnpc0000gn/T/ipykernel_5302/1747704286.py", line 35, in predict_fraud_with_rf
    fraud_probability = rf_model.predict_proba(new_transaction)[:, 1][0]
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/UZAIR/Desktop/fraud_detection/venv/lib/python3.12/site-packages/sklearn/ensemble/_forest.py", line 946, in predict_proba
    X = self._validate_X_predict(X)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/UZAIR/Desktop/fraud_detection/venv/lib/python3.12/site-packages/sklearn/ensemble/_forest.py", line 638, in _validate_X_predict
    X = validate_data(
        ^^^^^^^^^^^^^^
  File "/Users/UZAIR/Desktop/fraud_detection/venv/lib/python3.12/site-packages/sklearn/utils/validation.py", line 2944, in validate_data
    out = check_array(X, input_name="X", **check_params)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File

/Users/UZAIR/Desktop/fraud_detection/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


{'fraud_probability': 0.2, 'verdict': None, 'features_importance': None}